<a href="https://colab.research.google.com/github/jannikm81-gif/Humanitas-purissima-/blob/main/notebooks/living_constitution.ipyn" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time

class LivingConstitution:
    def __init__(self):
        # Ebene 0: Unveränderliche Kerninvarianten (Absolut geschützt, Wert: True)
        self.layer_0_invariants = {
            "unbedingte_privatsphaere": True,
            "noetigungsschutz_dkdp": True,
            "freiraum_garantie": True,
            "zero_utility_lock": True
        }

        # Ebene 1: Anpassbare Systemparameter (Über Bürgerräte modifizierbar)
        self.layer_1_parameters = {
            "subgrid_reserve_threshold": 200.0,
            "kl_decay_gamma": 0.3,
            "sabbatical_max_weeks": 26
        }

        # Warteschlange für vorgeschlagene Änderungen (Timelock-Register)
        self.timelock_registry = {}

        # 120 Tage in Sekunden konvertiert für die Protokoll-Invariante
        self.TIMELOCK_DURATION_SECONDS = 120 * 24 * 60 * 60

    def propose_parameter_change(self, parameter_name, new_value, is_layer_0=False):
        """
        Bürgerrat reicht einen Vorschlag ein.
        """
        current_timestamp = int(time.time())

        # Schutz-Invariante 1: Ebene 0 darf niemals angerührt werden
        if is_layer_0 or parameter_name in self.layer_0_invariants:
            return False, "PROTOKOLL-ALARM: Versuch der Manipulation von Ebene 0 abgewiesen!"

        # Wenn der Parameter in Ebene 1 existiert, trage ihn in das Timelock-Register ein
        if parameter_name in self.layer_1_parameters:
            execution_time = current_timestamp + self.TIMELOCK_DURATION_SECONDS
            self.timelock_registry[parameter_name] = {
                "new_value": new_value,
                "available_at": execution_time
            }
            return True, f"Vorschlag registriert. 120-Tage Timelock aktiv. Freischaltung am Zeitstempel: {execution_time}"

        return False, "Parameter nicht im Systemkontext vorhanden."

    def execute_parameter_change(self, parameter_name, simulated_current_time=None):
        """
        Führt die Änderung nach Ablauf des Timelocks aus.
        """
        if parameter_name not in self.timelock_registry:
            return False, "Keine registrierte Änderung für diesen Parameter gefunden."

        proposal = self.timelock_registry[parameter_name]
        actual_time = simulated_current_time if simulated_current_time is not None else int(time.time())

        # Schutz-Invariante 2: Überprüfung, ob die 120 Tage mathematisch abgelaufen sind
        if actual_time < proposal["available_at"]:
            remaining_days = (proposal["available_at"] - actual_time) / (24 * 60 * 60)
            return False, f"ABGEWIESEN: Timelock aktiv. Verbleibende Sperrzeit: {remaining_days:.1f} Tage."

        # Nach erfolgreichem Ablauf des Timelocks: Parameter wird aktualisiert
        self.layer_1_parameters[parameter_name] = proposal["new_value"]
        del self.timelock_registry[parameter_name]
        return True, f"ERFOLG: Parameter '{parameter_name}' wurde auf {new_value} aktualisiert."

# --- SIMULATION DER LIVING CONSTITUTION ---

constitution = LivingConstitution()
now = int(time.time())

print("=== HP-v10: LIVING CONSTITUTION SCHUTZ-AUDIT ===")
print("-" * 60)

# Test 1: Bösartiger Angriff auf die Grundrechte (Ebene 0)
print("TEST 1: Versuch, die unbedingte Privatsphäre abzuschalten...")
success_0, msg_0 = constitution.propose_parameter_change("unbedingte_privatsphaere", False)
print(f"Status: {success_0}")
print(f"System-Meldung: {msg_0}")
print("-" * 60)

# Test 2: Bürgerrat schlägt legitime Parameteränderung vor (Ebene 1)
print("TEST 2: Bürgerrat beantragt Erhöhung der maximalen Sabbatical-Wochen...")
success_1, msg_1 = constitution.propose_parameter_change("sabbatical_max_weeks", 52)
print(f"Status: {success_1}")
print(f"System-Meldung: {msg_1}")
print("-" * 60)

# Test 3: Illegaler Versuch der Sofort-Ausführung (Timelock brechen)
print("TEST 3: Versuch, die Änderung sofort auszuführen (nach nur 5 Tagen)...")
five_days_later = now + (5 * 24 * 60 * 60)
success_2, msg_2 = constitution.execute_parameter_change("sabbatical_max_weeks", simulated_current_time=five_days_later)
print(f"Status: {success_2}")
print(f"System-Meldung: {msg_2}")
print("-" * 60)

# Test 4: Legitime Ausführung nach Ablauf des 120-Tage Timelocks
print("TEST 4: Ausführung nach Ablauf von 121 Tagen...")
one_hundred_twenty_one_days_later = now + (121 * 24 * 60 * 60)
new_value = 52 # Referenzwert für die Ausgabe
success_3, msg_3 = constitution.execute_parameter_change("sabbatical_max_weeks", simulated_current_time=one_hundred_twenty_one_days_later)
print(f"Status: {success_3}")
print(f"System-Meldung: {msg_3}")
print("-" * 60)

=== HP-v10: LIVING CONSTITUTION SCHUTZ-AUDIT ===
------------------------------------------------------------
TEST 1: Versuch, die unbedingte Privatsphäre abzuschalten...
Status: False
System-Meldung: PROTOKOLL-ALARM: Versuch der Manipulation von Ebene 0 abgewiesen!
------------------------------------------------------------
TEST 2: Bürgerrat beantragt Erhöhung der maximalen Sabbatical-Wochen...
Status: True
System-Meldung: Vorschlag registriert. 120-Tage Timelock aktiv. Freischaltung am Zeitstempel: 1801945697
------------------------------------------------------------
TEST 3: Versuch, die Änderung sofort auszuführen (nach nur 5 Tagen)...
Status: False
System-Meldung: ABGEWIESEN: Timelock aktiv. Verbleibende Sperrzeit: 115.0 Tage.
------------------------------------------------------------
TEST 4: Ausführung nach Ablauf von 121 Tagen...
Status: True
System-Meldung: ERFOLG: Parameter 'sabbatical_max_weeks' wurde auf 52 aktualisiert.
--------------------------------------------------